In [ ]:
from moabb.datasets import PhysionetMI
from moabb.paradigms import MotorImagery
from moabb.evaluations import WithinSessionEvaluation

from sklearn.pipeline import make_pipeline
from sklearn.multiclass import OneVsRestClassifier
from sklearn.svm import SVC
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

from pyriemann.estimation import Covariances
from pyriemann.tangentspace import TangentSpace

from mne.decoding import CSP
from brainbot_dataset import get_brainbot_dataset

import moabb
import mne

moabb.set_log_level('INFO')
mne.set_log_level('INFO')

NUMBER_OF_TRIALS = 4

brainbot_dataset = get_brainbot_dataset()
brainbot_dataset.n_sessions = min(NUMBER_OF_TRIALS, brainbot_dataset.n_sessions)
physionet_dataset = PhysionetMI()
physionet_dataset.subject_list = physionet_dataset.subject_list[:NUMBER_OF_TRIALS]
assert len(physionet_dataset.subject_list) == NUMBER_OF_TRIALS
assert brainbot_dataset.n_sessions == NUMBER_OF_TRIALS

datasets = [brainbot_dataset, physionet_dataset]
dataset_results = {}
dataset_events = ["left_hand", "right_hand", "feet", "hands", "rest"]
sampling = 160 # based on Physionet sampling rate 

paradigm = MotorImagery(n_classes=len(dataset_events), events=dataset_events, resample=sampling)
# paradigm = LeftRightImagery()

pipelines = {}

# Base classifiers and preprocessing
svm = OneVsRestClassifier(SVC(kernel='rbf', probability=True))
csp = CSP(n_components=4, reg=None, log=True, norm_trace=False)

pipelines['CSP + SVM'] = make_pipeline(csp, svm)
pipelines['CSP + LDA'] = make_pipeline(CSP(n_components=8), LinearDiscriminantAnalysis())

# TGSP (Riemannian) pipeline
pipelines['TGSP + SVM'] = make_pipeline(Covariances("oas"), TangentSpace(metric="riemann"), SVC(kernel="linear", probability=True))

evaluation = WithinSessionEvaluation(paradigm=paradigm, datasets=datasets, overwrite=True)
results = evaluation.process(pipelines)

In [ ]:
print("Results Summary:")
summary = results.groupby(['pipeline', 'dataset'])['score'].agg(['mean', 'std', 'count'])
summary['mean'] = summary['mean'].round(3)
summary['std'] = summary['std'].round(3)
print(summary.to_string())
print("=" * 50)

print("\nDetailed Results by Subject and Dataset:")
detailed = results.pivot_table(
    index=['dataset', 'subject', 'session'], 
    columns='pipeline', 
    values='score'
)
print(detailed.round(3).to_string())
print("=" * 50)